# Notebook 3: Predictive Analysis

**Project:** How Time and Location Choices Change the Carbon Footprint of AI Training  
**Data:** EIA Form 930 BALANCE — Jan–Dec 2025  
**Author:** Lin Htet Aung — MSc Business Analytics, Trinity College Dublin

### Research Questions Addressed
- **RQ3/RQ4:** Can carbon exposure be reliably forecast for operational scheduling use?
- Which model provides actionable value beyond simple lag-based rules?

### Models compared
| Model | Notes |
|-------|-------|
| Naive baseline (lag-1h) | Persistence: predict next hour = current hour |
| Linear Regression | Baseline ML |
| **Random Forest** ✓ | **Selected model** — best MAE, interpretable |
| XGBoost | Competitive performance |
| Day-ahead RF | No lag-1h — real scheduling scenario |

---
**Prerequisites:** Run `01_data_prep.ipynb` first.

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/analysis_dataset.csv', low_memory=False)
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 2. Feature Engineering & Train/Test Split

**Chronological split** (Jan–Oct training, Oct–Dec test) to avoid data leakage on autocorrelated series.

In [ ]:
# Drop rows missing any required feature
required_cols = [
    'carbon_proxy',
    'carbon_proxy_lag1h',
    'carbon_proxy_lag24h',
    'carbon_proxy_lag168h',
    'demand_lag1h'
]
df = df.dropna(subset=required_cols)
print(f'After dropna: {len(df):,} rows')

# Features used in short-term model
features = [
    'hour_local',
    'weekday',
    'month',
    'demand_mw',
    'carbon_proxy_lag1h',
    'carbon_proxy_lag24h',
    'carbon_proxy_lag168h',
    'demand_lag1h'
]

# Chronological split: last 3 months as test set
df['utc_datetime'] = pd.to_datetime(df['utc_datetime'])
cutoff = df['utc_datetime'].max() - pd.DateOffset(months=3)
train = df[df['utc_datetime'] <= cutoff]
test  = df[df['utc_datetime'] > cutoff]

X_train = train[features]
y_train = train['carbon_proxy']
X_test  = test[features]
y_test  = test['carbon_proxy']

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')
print(f'Test period: {test["utc_datetime"].min().date()} → {test["utc_datetime"].max().date()}')

## 3. Model Training & Evaluation

### 3.1 Naive Persistence Baseline (lag-1h)

In [ ]:
y_naive = test['carbon_proxy_lag1h']
mae_naive = mean_absolute_error(y_test, y_naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, y_naive))
print(f'Naive (lag-1h) — MAE: {mae_naive:.4f}  RMSE: {rmse_naive:.4f}')

### 3.2 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f'Linear Regression — MAE: {mae_lr:.4f}  RMSE: {rmse_lr:.4f}')

### 3.3 Random Forest (Selected Model)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f'Random Forest — MAE: {mae_rf:.4f}  RMSE: {rmse_rf:.4f}')

### 3.4 XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print(f'XGBoost — MAE: {mae_xgb:.4f}  RMSE: {rmse_xgb:.4f}')

### 3.5 Day-Ahead Random Forest (No lag-1h — Real Scheduling Scenario)

In [ ]:
# Remove lag-1h and contemporaneous demand — unavailable 24h ahead
features_da = [
    'hour_local', 'weekday', 'month',
    'carbon_proxy_lag24h', 'carbon_proxy_lag168h'
]
rf_da = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_da.fit(train[features_da], y_train)
y_pred_da = rf_da.predict(test[features_da])

mae_da  = mean_absolute_error(y_test, y_pred_da)
rmse_da = np.sqrt(mean_squared_error(y_test, y_pred_da))

# Day-ahead naive baseline (lag-24h)
y_da_naive = test['carbon_proxy_lag24h']
mae_da_naive = mean_absolute_error(y_test, y_da_naive)

print(f'Day-ahead Naive (lag-24h) — MAE: {mae_da_naive:.4f}')
print(f'Day-ahead RF (no lag-1h) — MAE: {mae_da:.4f}')

## 4. Results Summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'Naive (lag-1h)',       'MAE': mae_naive,    'RMSE': rmse_naive,  'vs Naive': '—'},
    {'Model': 'Linear Regression',    'MAE': mae_lr,       'RMSE': rmse_lr,     'vs Naive': f"{(mae_naive - mae_lr)/mae_naive*100:+.1f}%"},
    {'Model': 'Random Forest ✓',      'MAE': mae_rf,       'RMSE': rmse_rf,     'vs Naive': f"{(mae_naive - mae_rf)/mae_naive*100:+.1f}%"},
    {'Model': 'XGBoost',              'MAE': mae_xgb,      'RMSE': rmse_xgb,    'vs Naive': f"{(mae_naive - mae_xgb)/mae_naive*100:+.1f}%"},
    {'Model': 'Day-ahead Naive (24h)','MAE': mae_da_naive, 'RMSE': float('nan'),'vs Naive': '—'},
    {'Model': 'Day-ahead RF',         'MAE': mae_da,       'RMSE': float('nan'),'vs Naive': f"{(mae_da_naive - mae_da)/mae_da_naive*100:+.1f}% vs lag-24h"},
])

print('=== Model Comparison ===')
print(results.to_string(index=False, float_format='{:.4f}'.format))

## 5. Model Comparison Chart

In [ ]:
models     = ['Naive\n(lag-1h)', 'Linear\nRegression', 'Random\nForest', 'XGBoost']
mae_values = [mae_naive, mae_lr, mae_rf, mae_xgb]
colors     = ['#9E9E9E', '#2196F3', '#4CAF50', '#F44336']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, mae_values, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
ax.axhline(mae_naive, color='black', linestyle='--', linewidth=1, label='Naive baseline')
ax.set_ylabel('MAE (lower is better)')
ax.set_title('Model Comparison — Test Set MAE')
ax.legend()

for bar, val in zip(bars, mae_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.0002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/fig8_model_comparison.png', bbox_inches='tight')
plt.show()

## 6. Random Forest Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi['feature'], fi['importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importance')

plt.tight_layout()
plt.savefig('../outputs/figures/fig9_feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop feature shares:')
for _, row in fi.sort_values('importance', ascending=False).iterrows():
    print(f'  {row["feature"]:<30} {row["importance"]*100:.1f}%')

## 7. Predicted vs Actual (Random Forest)

In [ ]:
sample_idx = np.random.choice(len(y_test), size=5000, replace=False)
y_actual_sample   = y_test.values[sample_idx]
y_pred_rf_sample  = y_pred_rf[sample_idx]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_actual_sample, y_pred_rf_sample, alpha=0.3, s=6, color='steelblue')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual carbon_proxy')
ax.set_ylabel('Predicted carbon_proxy')
ax.set_title('Predicted vs Actual — Random Forest (5,000-point sample)')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/fig_predicted_vs_actual.png', bbox_inches='tight')
plt.show()

## 8. Key Insights

1. **Random Forest** achieves the best short-term MAE (0.0191), 4.7% better than the naive persistence baseline.

2. **`carbon_proxy_lag1h` dominates** feature importance (~98.9%). Fossil dispatch changes slowly because thermal plants have high ramping costs — the grid one hour ago is the best predictor of the grid right now.

3. **Day-ahead forecasting is hard.** Removing `lag1h` causes a 2.7× increase in MAE (0.0191 → 0.0517). The day-ahead RF performs *worse* than the trivial lag-24h baseline in Texas, Central, and Southwest — the very regions where time-shifting offers the most value.

4. **Practical implication:** Use rolling short-term RF forecasts for real-time advisory, but base scheduling strategy on empirical regional carbon profiles, not ML predictions alone.